In [1]:
import os
print(os.getcwd())  # Must print the project_root path

c:\Users\yashg\edtech-nlp-pipeline


In [2]:
import json

with open('data/raw/pii/train.json') as f:
    data = json.load(f)

print(type(data))
print(data[0].keys())
print(data[0]['tokens'][:10])
print(data[0]['labels'][:10])
print(f"Total documents: {len(data)}")

<class 'list'>
dict_keys(['document', 'full_text', 'tokens', 'trailing_whitespace', 'labels'])
['Design', 'Thinking', 'for', 'innovation', 'reflexion', '-', 'Avril', '2021', '-', 'Nathalie']
['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-NAME_STUDENT']
Total documents: 6807


In [3]:
from collections import Counter

counter = Counter()
for doc in data:
    counter.update(doc['labels'])

total = sum(counter.values())
print("\nLabel distribution:")
for label, count in counter.most_common():
    print(f"  {label:<25} {count:>8}  ({100*count/total:.2f}%)")


Label distribution:
  O                          4989794  (99.95%)
  B-NAME_STUDENT                1365  (0.03%)
  I-NAME_STUDENT                1096  (0.02%)
  B-URL_PERSONAL                 110  (0.00%)
  B-ID_NUM                        78  (0.00%)
  B-EMAIL                         39  (0.00%)
  I-STREET_ADDRESS                20  (0.00%)
  I-PHONE_NUM                     15  (0.00%)
  B-USERNAME                       6  (0.00%)
  B-PHONE_NUM                      6  (0.00%)
  B-STREET_ADDRESS                 2  (0.00%)
  I-URL_PERSONAL                   1  (0.00%)
  I-ID_NUM                         1  (0.00%)


In [4]:
unique_labels = sorted(set(l for doc in data for l in doc['labels']))
print("\nUnique labels:")
print(unique_labels)
print(f"\nTotal unique labels: {len(unique_labels)}")
# Copy this output into a comment at top of src/data/pii_dataset.py


Unique labels:
['B-EMAIL', 'B-ID_NUM', 'B-NAME_STUDENT', 'B-PHONE_NUM', 'B-STREET_ADDRESS', 'B-URL_PERSONAL', 'B-USERNAME', 'I-ID_NUM', 'I-NAME_STUDENT', 'I-PHONE_NUM', 'I-STREET_ADDRESS', 'I-URL_PERSONAL', 'O']

Total unique labels: 13


In [5]:
mismatches = []
for doc in data:
    if len(doc['tokens']) != len(doc['labels']):
        mismatches.append(doc['document'])

if mismatches:
    print(f"MISMATCH in {len(mismatches)} documents: {mismatches}")
else:
    print("All docs clean.")

All docs clean.


In [1]:
import json
from transformers import AutoTokenizer
from src.data.pii_dataset import load_pii_records, PIIDataset, LABEL2ID, ID2LABEL

tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base", use_fast=True)

with open("data/raw/pii/train.json") as f:
    raw = json.load(f)

# Use the first doc (we know doc[0] has B-NAME_STUDENT at position 9 from T0)
test_doc    = raw[0]
words       = test_doc['tokens'][:15]
word_labels = test_doc['labels'][:15]

encoding = tokenizer(words, is_split_into_words=True,
                     max_length=64, truncation=True, padding='max_length',
                     return_tensors='pt')
word_ids = encoding.word_ids(batch_index=0)
tokens   = tokenizer.convert_ids_to_tokens(encoding['input_ids'][0])

print(f"{'word_id':<8}| {'subword_token':<27}| assigned_label")
print("-" * 70)
prev_wid = None
for tok, wid in zip(tokens, word_ids):
    if wid is None:
        label = "-100 (special)"
    elif wid != prev_wid:
        label = f"{word_labels[wid]} (FIRST subword)"
    else:
        label = "-100 (continuation)"
    print(f"{str(wid):<8}| {tok:<27}| {label}")
    prev_wid = wid

# Sanity check: no B- or I- label on a continuation row

c:\Users\yashg\edtech-nlp-pipeline\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\yashg\edtech-nlp-pipeline\venv\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
c:\Users\yashg\edtech-nlp-pipeline\venv\lib\site-packages\transformers\convert_slow_tokenizer.py:550: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into 

word_id | subword_token              | assigned_label
----------------------------------------------------------------------
None    | [CLS]                      | -100 (special)
0       | ▁Design                    | O (FIRST subword)
1       | ▁Thinking                  | O (FIRST subword)
2       | ▁for                       | O (FIRST subword)
3       | ▁innovation                | O (FIRST subword)
4       | ▁reflex                    | O (FIRST subword)
4       | ion                        | -100 (continuation)
5       | ▁-                         | O (FIRST subword)
6       | ▁Avril                     | O (FIRST subword)
7       | ▁2021                      | O (FIRST subword)
8       | ▁-                         | O (FIRST subword)
9       | ▁Nathalie                  | B-NAME_STUDENT (FIRST subword)
10      | ▁S                         | I-NAME_STUDENT (FIRST subword)
10      | ylla                       | -100 (continuation)
12      | ▁Challenge                 | O (FIRST su

In [2]:
records = load_pii_records("data/raw/pii/train.json")
ds      = PIIDataset(records[:5], tokenizer, LABEL2ID, max_length=512)
item    = ds[0]

print(item['input_ids'].shape)                       # torch.Size([512])
print(item['attention_mask'].shape)                  # torch.Size([512])
print(item['labels'].shape)                          # torch.Size([512])
print((item['labels'] != -100).sum().item())         # Number of word-level tokens

assert item['input_ids'].shape == (512,)
assert item['labels'].shape    == (512,)
print("Shape assertions passed.")

torch.Size([512])
torch.Size([512])
torch.Size([512])
486
Shape assertions passed.


In [3]:
import torch
from src.models.pii_model import PIITokenClassifier
from src.data.pii_dataset import LABEL2ID

print(f"num_labels = {len(LABEL2ID)}")   # Must be 13

model = PIITokenClassifier("microsoft/deberta-v3-base", num_labels=len(LABEL2ID))
model.enable_gradient_checkpointing()

B, L = 2, 512
out  = model(input_ids=torch.randint(0, 1000, (B, L)),
             attention_mask=torch.ones(B, L, dtype=torch.long))

print(out['logits'].shape)               # Expected: torch.Size([2, 512, 13])
assert out['logits'].shape == (B, L, len(LABEL2ID))
print("Forward pass: OK")

num_labels = 13
torch.Size([2, 512, 13])
Forward pass: OK


In [1]:
from src.evaluation.pii_metrics import compute_entity_f1

# Test 1: perfect → F1 must be 1.0
true = [["O", "B-NAME_STUDENT", "I-NAME_STUDENT", "O"]]
pred = [["O", "B-NAME_STUDENT", "I-NAME_STUDENT", "O"]]
r    = compute_entity_f1(true, pred)
assert r['entity_f1'] == 1.0, f"Expected 1.0, got {r['entity_f1']}"
print(f"Perfect: F1={r['entity_f1']}")

# Test 2: all-O on non-O target → F1 must be 0.0
r2 = compute_entity_f1(true, [["O", "O", "O", "O"]])
assert r2['entity_f1'] == 0.0, f"Expected 0.0, got {r2['entity_f1']}"
print(f"Missed entity: F1={r2['entity_f1']}")

print("All tests passed.")

Perfect: F1=1.0
Missed entity: F1=0.0
All tests passed.


c:\Users\yashg\edtech-nlp-pipeline\venv\lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Users\yashg\edtech-nlp-pipeline\venv\lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [1]:
import torch
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import LambdaLR
from torch.cuda.amp import GradScaler
from transformers import AutoTokenizer
from src.data.pii_dataset import load_pii_records, train_val_split, PIIDataset, LABEL2ID
from src.models.pii_model import PIITokenClassifier
from src.training.pii_trainer import compute_class_weights, train_one_epoch

tokenizer  = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base", use_fast=True)
records    = load_pii_records("data/raw/pii/train.json")
train_recs, _ = train_val_split(records, val_fraction=0.2, random_state=42)

# Use 50 documents — enough to populate all 13 labels in the counter
ds      = PIIDataset(train_recs[:50], tokenizer, LABEL2ID, max_length=128)
loader  = DataLoader(ds, batch_size=2, shuffle=False, num_workers=0)

device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model   = PIITokenClassifier("microsoft/deberta-v3-base",
                              num_labels=len(LABEL2ID)).to(device)
model.enable_gradient_checkpointing()

weights   = compute_class_weights(train_recs[:50], LABEL2ID, device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
scheduler = LambdaLR(optimizer, lambda step: 1.0)
scaler    = GradScaler()

# Verify weights are sane
print("Class weights:")
from src.data.pii_dataset import PII_LABELS
for lbl, w in zip(PII_LABELS, weights.cpu().tolist()):
    print(f"  {lbl:<25} {w:.3f}")

assert all(torch.isfinite(weights)), "Weights contain inf or nan!"
assert weights.max().item() <= 50.0, f"Max weight {weights.max():.1f} exceeds cap!"
print("Weight assertions passed.")

loss = train_one_epoch(model, loader, optimizer, scheduler, scaler, device, weights)
print(f"\nSmoke test loss: {loss:.4f}")
assert torch.isfinite(torch.tensor(loss)), "Loss is nan or inf!"
print("Smoke test passed.")

c:\Users\yashg\edtech-nlp-pipeline\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\yashg\edtech-nlp-pipeline\venv\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
c:\Users\yashg\edtech-nlp-pipeline\venv\lib\site-packages\transformers\convert_slow_tokenizer.py:550: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into 

Class weights:
  O                         0.100
  B-NAME_STUDENT            4.819
  I-NAME_STUDENT            4.982
  B-EMAIL                   9.109
  B-USERNAME                9.109
  B-ID_NUM                  6.912
  I-ID_NUM                  9.109
  B-PHONE_NUM               9.109
  I-PHONE_NUM               9.109
  B-URL_PERSONAL            7.723
  I-URL_PERSONAL            9.109
  B-STREET_ADDRESS          9.109
  I-STREET_ADDRESS          9.109
Weight assertions passed.

Smoke test loss: 2.2337
Smoke test passed.


In [2]:
from src.utils.config import load_config
cfg = load_config("configs/config.yaml")

assert cfg['stage1']['num_labels'] == 13,    f"Expected 13, got {cfg['stage1']['num_labels']}"
assert cfg['stage1']['batch_size'] == 8,     "batch_size wrong"
assert cfg['stage1']['class_weight_cap'] == 50.0, "cap wrong"
print("Config OK.")
print(cfg['stage1'])

Config OK.
{'train_path': 'data/raw/pii/train.json', 'external_records_path': 'data/raw/pii/external_records.json', 'val_fraction': 0.2, 'random_state': 42, 'model_name': 'microsoft/deberta-v3-base', 'max_length': 512, 'dropout': 0.1, 'num_labels': 13, 'num_epochs': 3, 'batch_size': 8, 'learning_rate': 2e-05, 'warmup_fraction': 0.1, 'max_grad_norm': 1.0, 'class_weight_cap': 50.0, 'max_external_records': 5000, 'inference_threshold': 0.5, 'model_save_dir': 'outputs/models/pii'}


In [ ]:
import pandas as pd
log = pd.read_csv("outputs/experiment_log.csv", encoding='utf-8-sig')
print(log[['experiment_id', 'stage', 'metric_after', 'verdict']].to_string())